# 04 — Mock Protocol -> PLPS Generation Pipeline with Grounding Check (from scratch, offline)

Companion notebook to `../05-building-instruction-based-generation-pipelines.md`.

This notebook chains together two stages of the real generation pipeline, both fully mocked and
offline:

1. A **mock instruction-based generation step** — simple, deterministic, template-based text
   generation standing in for an instruction-tuned LLM call, producing a Plain Language Protocol
   Synopsis (PLPS) sentence from a source Protocol section.
2. A **factual-grounding check** — verifies that the quantities (ages, doses, weights,
   frequencies) and any strong clinical-outcome claims in the generated text are actually
   supported by the source section, flagging anything that isn't.

No LLM call, no API key, no model download — pure Python standard library (`re` only).

## 1. Source protocol sections (synthetic)

Two short excerpts standing in for structured sections produced by the parsing stage in
notebook 01 — an eligibility criteria section and a dosing section.

In [ ]:
SOURCE_SECTIONS = {
    "3.1": {
        "title": "Inclusion Criteria",
        "text": (
            "Adults aged 18 to 75 years, inclusive, at the time of screening. Confirmed diagnosis "
            "of moderate to severe persistent asthma for at least 12 months. Pre-bronchodilator "
            "FEV1 of 40 to 90 percent of predicted normal value at screening."
        ),
    },
    "5.1": {
        "title": "Dose and Regimen",
        "text": (
            "Participants weighing less than 60 kg will receive 100 mg subcutaneously every 4 "
            "weeks. Participants weighing 60 kg or more will receive 200 mg subcutaneously every "
            "4 weeks."
        ),
    },
}

for sid, s in SOURCE_SECTIONS.items():
    print(f"[{sid}] {s['title']}: {s['text'][:70]}...")

## 2. Mock instruction-based generation

This is **not** a real LLM call — it's a deterministic template filled in with values drawn from
the source section, standing in for what a properly grounded instruction-tuned model call
*should* produce. Section 4 below shows what the grounding check does when a generation drifts
from this ideal (as a real model occasionally will).

In [ ]:
TEMPLATES = {
    "plps_eligibility": "To take part in this study, you must be {age_range} years old and have {diagnosis}.",
    "plps_dosing": "You will get a shot under your skin every {frequency}. The dose depends on your body weight: {weight_rule}.",
}

def mock_generate_plps(section_id: str):
    """Deterministic, template-based mock generator standing in for an instruction-tuned LLM
    call (see Chapter 05 for the real prompt-template design). Values are pulled FROM the source
    section, so this "clean" path is grounded by construction."""
    section = SOURCE_SECTIONS[section_id]
    if section_id == "3.1":
        return TEMPLATES["plps_eligibility"].format(
            age_range="18 to 75",
            diagnosis="moderate to severe persistent asthma for at least 12 months",
        )
    if section_id == "5.1":
        return TEMPLATES["plps_dosing"].format(
            frequency="4 weeks",
            weight_rule="100 mg if you weigh less than 60 kg, 200 mg if you weigh 60 kg or more",
        )
    raise KeyError(section_id)

for sid in SOURCE_SECTIONS:
    gen = mock_generate_plps(sid)
    print(f"[{sid}] {SOURCE_SECTIONS[sid]['title']}")
    print("  generated:", gen)

## 3. Factual-grounding check

The naive approach — flag every word in the generated text that doesn't literally appear in the
source — produces **false positives** on ordinary, desirable paraphrasing: a PLPS is *supposed*
to say "a shot under the skin" instead of "subcutaneously," and that vocabulary swap is not a
hallucination. What actually must never drift between source and generated text in a document
like this are:

- **Quantities** — ages, doses, weights, frequencies, percentages. A generated PLPS that says
  "every 2 weeks" when the source says "every 4 weeks" is a dangerous, silent factual error, even
  though it's a tiny, easy-to-miss token-level change.
- **Strong outcome claims** — words like "cure," "guarantee," "completely," "eliminate" assert
  something about efficacy or safety that a Protocol excerpt like this would essentially never
  itself state in that language; if the model produces one and the source doesn't, that's exactly
  the class of fabricated clinical claim Chapter 05's grounding defenses exist to catch.

The check below implements both, and *deliberately* does not flag general vocabulary
differences — matching the "verify key terms/entities, not verify every word" approach described
in the chapter.

In [ ]:
import re

# Layer 1: numeric/quantity claims -- must be exactly reproduced from the source
QUANTITY_RE = re.compile(r"\b(\d+(?:\.\d+)?)\s*(kg|mg|percent|%|weeks?|days?|years?|months?)\b", re.IGNORECASE)

def extract_quantities(text: str):
    return {(num, unit.lower().rstrip("s")) for num, unit in QUANTITY_RE.findall(text)}

# Layer 2: watchlist of strong, unsupportable clinical-outcome claim words
CLAIM_WATCHLIST = {
    "cure", "cures", "cured", "guarantee", "guaranteed", "eliminate", "eliminates",
    "completely", "always", "never", "risk-free", "painless", "harmless",
}

def watchlist_hits(text: str):
    tokens = re.findall(r"[a-z\-]+", text.lower())
    return sorted(set(tokens) & CLAIM_WATCHLIST)

def grounding_check(generated_text: str, source_text: str):
    """Flag (a) quantities in the generated text absent from the source, and (b) any strong
    outcome-claim word not itself present in the source. Deliberately tolerant of ordinary
    paraphrasing; focused on the two failure modes that matter most in a regulated document."""
    gen_quantities = extract_quantities(generated_text)
    source_quantities = extract_quantities(source_text)
    unsupported_quantities = sorted(gen_quantities - source_quantities)

    gen_claims = set(watchlist_hits(generated_text))
    source_claims = set(watchlist_hits(source_text))
    unsupported_claims = sorted(gen_claims - source_claims)

    clean = not unsupported_quantities and not unsupported_claims
    return {
        "clean": clean,
        "unsupported_quantities": unsupported_quantities,
        "unsupported_claims": unsupported_claims,
    }

print("--- Grounding checks on the clean, template-generated text ---")
for sid in SOURCE_SECTIONS:
    gen = mock_generate_plps(sid)
    result = grounding_check(gen, SOURCE_SECTIONS[sid]["text"])
    print(f"[{sid}] clean={result['clean']}  "
          f"unsupported_quantities={result['unsupported_quantities']}  "
          f"unsupported_claims={result['unsupported_claims']}")
    assert result["clean"], f"expected clean grounding for {sid}, got {result}"

print("\nOK: both clean generations pass the grounding check, with zero unsupported quantities "
      "or claims -- as expected, since they were built from the source values.")

## 4. What happens when a generation drifts (the failure case)

A hand-written, deliberately corrupted "generation" standing in for what an ungrounded LLM call
might produce: the dosing frequency has silently changed from 4 weeks to 2 weeks, and an
unsupported efficacy claim ("completely cure") has been introduced. This is exactly the kind of
error the grounding check exists to catch before it ever reaches a human reviewer's screen.

In [ ]:
hallucinated = "You will get a shot under your skin every 2 weeks, and it will completely cure your asthma."

result = grounding_check(hallucinated, SOURCE_SECTIONS["5.1"]["text"])
print("generated: ", hallucinated)
print("source:    ", SOURCE_SECTIONS["5.1"]["text"])
print()
print("clean:                ", result["clean"])
print("unsupported_quantities:", result["unsupported_quantities"])
print("unsupported_claims:    ", result["unsupported_claims"])

assert not result["clean"]
assert ("2", "week") in result["unsupported_quantities"]
assert "cure" in result["unsupported_claims"] and "completely" in result["unsupported_claims"]
print("\nOK: the check flags both the drifted frequency (2 weeks vs. the source's 4 weeks) and "
      "the fabricated efficacy claim -- exactly the kind of error that must never reach a "
      "patient-facing document without being caught first.")

## 5. The end-to-end mock pipeline

Chaining generation and the grounding check into one function, with a routing decision at the
end — mirroring the shape of the real pipeline in Chapter 05 (generate -> grounding check ->
reviewer queue), minus the real model call and the real human reviewer.

In [ ]:
def run_pipeline(section_id: str, override_generation: str = None):
    """generate (or use a supplied override) -> grounding check -> routing decision."""
    source = SOURCE_SECTIONS[section_id]
    generated = override_generation if override_generation is not None else mock_generate_plps(section_id)
    check = grounding_check(generated, source["text"])
    status = "CLEAN_DRAFT_FOR_REVIEW" if check["clean"] else "FLAGGED_FOR_REVIEW"
    return {
        "section_id": section_id,
        "source_title": source["title"],
        "generated_text": generated,
        "grounding": check,
        "status": status,
    }

print("--- Full pipeline runs, clean generations ---")
for sid in SOURCE_SECTIONS:
    result = run_pipeline(sid)
    print(f"[{sid}] status={result['status']}")
    assert result["status"] == "CLEAN_DRAFT_FOR_REVIEW"

print("\n--- Full pipeline run, corrupted generation ---")
corrupted_result = run_pipeline("5.1", override_generation=hallucinated)
print(f"[5.1, corrupted] status={corrupted_result['status']}")
print(f"  unsupported_quantities={corrupted_result['grounding']['unsupported_quantities']}")
print(f"  unsupported_claims=    {corrupted_result['grounding']['unsupported_claims']}")
assert corrupted_result["status"] == "FLAGGED_FOR_REVIEW"

print("\nAll checks passed.")

## 6. Tying it back

- This pipeline is a toy-scale version of exactly what Chapter 05 describes: a per-section-type
  generation step, followed immediately by an automated grounding check, followed by a routing
  decision (`CLEAN_DRAFT_FOR_REVIEW` vs. `FLAGGED_FOR_REVIEW`) — **never** a routing decision of
  "auto-publish."
- Both outcomes here still say "FOR_REVIEW" — even a clean grounding check result is a *draft*,
  not a deliverable, consistent with the human-in-the-loop principle running through this whole
  course.
- The quantity/claim-focused grounding check design (rather than flagging every vocabulary
  difference) is a deliberate, defensible choice worth naming in an interview: it's precise
  enough to catch the failure modes that actually matter in a regulated document (drifted
  numbers, invented outcome claims) while staying tolerant of the paraphrasing a plain-language
  rewrite is supposed to do.